# Gemma 4 Developer Agent — Phase 0 checks, EXP-000 gold control, baseline eval (4x L4)

One notebook, three switches (`RUN_PHASE0_CHECKS`, `RUN_GOLD_CONTROL`, `RUN_AGENT`). It mirrors the
organizers' Getting-Started recipe (wheelhouse install, `VllmServer`, `swegemma.evaluate.Evaluator` with the
subprocess sandbox) and writes everything under `/kaggle/working/runs/<EXP_ID>/` in the layout that
`tools/analyze_traces.py` and `tools/estimate_budget.py` read.

**Attach as inputs** (Add Input): competition `gemma-4-developer-agent`, dataset
`metric/gemma-4-developer-agent-wheelhouse`, model `google/gemma-4/Other/gemma-4-31b-it-qat-w4a16-ct/2`
(or the `.../gemma-4-e4b-it-qat-w4a16-ct/2` / `.../gemma-4-e2b-it-qat-w4a16-ct/2` input when the zip's agent.yaml names a
small variant; the vLLM cell takes the model from the zip).
**Accelerator**: NVIDIA L4 x4 (competition-attached notebooks only). **Internet**: off.
The submission zip comes either from a private Kaggle dataset you upload (`SUBMISSION_ZIP`) or embedded as
base64 by `python tools/build_notebook.py --embed dist/submission.zip`.

**Status: UNTESTED on Kaggle.** This file was written without access to kaggle.com, the wheelhouse or the
harness source. Cells marked `# VERIFY` use API names taken from a community mirror of the starter notebook
and may need adjusting against the real `getting-started-gemma-4-developer-agent` notebook. Read the
starter notebook first and diff the server/evaluator cells.

Edit the parameters in the next cell, save a notebook *version* per experiment, then download
`/kaggle/working/runs/<EXP_ID>.zip` and run `python tools/analyze_traces.py runs/<EXP_ID>`.

In [ ]:
# ---------------------------------------------------------------- parameters (edit me)
EXP_ID = "EXP-040-v1"   # overridden by --set
RUN_PHASE0_CHECKS = True          # Q1/Q2/Q3/Q7/Q9 answers + HARNESS_README copy + sample_submission dump
RUN_GOLD_CONTROL = False   # overridden by --set
RUN_AGENT = True                  # run our submission on the sound tasks
MEASURE_DECODE_SPEED = True       # Q10: one timed completion against the server

# Task selection: explicit ids win; otherwise the first N_TASKS of the (optionally repo-filtered) dataset.
TASK_IDS = ["fastapi_14301", "fastapi_14448", "fastapi_14605", "httpx_3672", "requests_7205", "requests_7309", "rich_3006", "rich_3468"]   # overridden by --set
N_TASKS = 8   # overridden by --set
REPO_FILTER = ""                  # e.g. "Textualize/rich"; "" = all repos
EXCLUDE_IDS = []                  # known-unsound tasks from a previous gold control (experiments/README.md Q5)

# Where the submission comes from (first that exists wins). Upload dist/submission.zip as a private dataset,
# or embed it: python tools/build_notebook.py --embed dist/submission.zip
SUBMISSION_ZIP = "/kaggle/input/datasets/YOUR_KAGGLE_USERNAME/gemma4-sub-v1/submission.zip"   # VERIFY path
SUBMISSION_B64 = "UEsDBBQAAAAIAAAAIVzWj3oRmgEAANICAAAKAAAAYWdlbnQueWFtbIWRQW/TQBCF7/kVI/WSStlgJyYHcwLUG4cKEEJCVTS1x86K9Y47O66bf8/YbpuKA1wseffN2/e9uYKbn7cuKzLYZbuDy3OXvS9BmBWqE8aWEjTCHXAk+BK6jy1FhbWeaJnb7+Gm+ATJxzYQBOb+GpThGz0MJvQY5oFfvXDPiWQDtygYAj2f5wd4ZCVJdxvASlnuYP3ZZbv9BlSI4Ed+vV1dwXd7bomEIt4iRYaOawomYw4JWMDHpDJU6jl+AHokOV/y+mTXkIb7I07/6R1grF/NvJrBGBdHWLfUdegKR8W98+oeUN1YYH5wlVrI9MJtwSJ2VEIaabFdzd9jFTCl8u8KVjWlSnw/5SvhK2ENU4lC1os38DOMXk+AdoK14xjOcCktkFrFT0oUZ2AXKKXn5gAVat80JBOpUteToA5iYL2vfs+L42aGNBuboIk81r5GpbSZUpip7eQ1QeOfTKPe0g2BjLnv7XJKO/oYSebyrMzO6zIyPTFvb7u6dFyuABxUHBvfHnvUU/l2AS9s2zN24Z/KifL/quX5WfYHUEsDBBQAAAAIAAAAIVz89GWVbAAAAIEAAAAVAAAAY29uZmlncy9zYW1wbGluZy55YW1sPcxBDoMgEEbhPaeYExhsrClzGWJ1CoQKhP6TePwaF+5evsWD7E36Au3CNA7WoDbfmOzgnldnpnky+3L4qmgKj5ql/Jheo3sYxFRyKsGvtXxSYEN001u3IGCarJtPT2X96iYesWqIOA/oKuYPUEsDBBQAAAAIAAAAIVynzIDmhQEAAHsCAAAeAAAAY29uZmlncy9zYW1wbGluZ19wcm9wb3Nlci55YW1sZZFPb5wwFMTvfIqRckhXAgSErBYi9daeK/XSG/LCAyyMbeFHNvn2fXjTqGpPGP+Z38ybB3z79SMr6gJVUZ2zssyK5xZ+c94F2uCsec/xU60EptXTpnjfKAU73/n7Z4EKCGr1Rtspf1ereQHP2i7y2133YSJGXTRnZF9RFlWN4KCwkRqyQx0iaKFFYnYb48tE9qBoFzdXFxhuxE0Zk96tVhcE6uOF5pQmD1jVW+d29jt37BayAZeyqQ5cxAqOZ8KorTKC9ZGykGfcZmcI2sbzV8e0hccj+uo5nA664rsBdRV91GlZFPhA0CuJwTE+Xd1A5jHI2u3TzDKpN4YhtQQRZwctwGj96amV6HV6uZRZ1EEvMEMxzO0YI5GNmZQdYB0jrBKcthRX6tUeSJ73ygtJrPG2214xBTkc3UbRi4hJDhpgtBgnO4TPdH9MNPhenvLkrz5blHmRxE5bFHnzHNdLi3Od/DfdNo41+ay4d3bUU5vg39bbWLfsa9ubfaDuYz6iINYp+Q1QSwMEFAAAAAgAAAAhXJmYG8QRAQAAlAEAABAAAABldmFsX2NvbmZpZy55YW1sPZDLbsIwEEX3+YorsSioJHIC5ZFFd+y76KK7yHUGEuF4Ij8S+PvaICp5MT4z985jgdPPVy62ApWodnlZ5uKjxiBvjQ/WOBwE8s+YEwVOE9k7Bm5JgyYyHnxOQYTykr6Kg/EOMj4kNZZupgsNg4wp4+nmi/FeV/t9Xh2PqzUm9mQdeqN0aKmtswXYEEbpIowmvxw8Dhgtj+zI4h3l7ilKoYBUnu2jk1vDcZp15qBbkGnTCNJdIc+p3M/8sCWH5WtfR8r3bFCtoTpSV6hVge+OwL5Lks4SYZI6RJG09LzTZvPmiowSlkldZ4DvB4qTNtGQTetqVFsR8eOGzLpRUutEDy8Y65uhN8FTxP80rVGnS2d/UEsDBBQAAAAIAAAAIVyLolMBjQgAAJwSAAAQAAAAcHJvbXB0cy9hY3Rvci5tZG1Y33PbNhJ+11+x4z6k7Ui05aSt43u4SdxkLtObJpP47tonESIhCWcSYAEwiq7X/73fLkBQ9vTFJkFgf3y7++1Cv7qRlNekLOkvg/aR1Biddb0bAwW3i0f+qu3eWK19RR91cN1nTfGgaQzakwlh1HQ08SBroVddp0PE2dE3mpqDsnverrCkogk7o0PaGVXULW31QX02zleLxT1WD6ZttaW9Vy2Eq2HoeP8JwmhvIrVmt6PoSNHO63AgrwcXTHT+BA9a8qMNZGIgd7QUYUao6F0kqz9DWNCTJK9VcNbYfUWv4Hc/xFOSHBoHsfQ/7R0+Ed68boov51vYg6jCA0G0JbNLgjvXqE4U01EF6GnJef6UTfCAWcl3uPvVV/R6bPc6Ln7FBqCg6foGzrmOIKUL4tEN9caOOFHRfw7QpAE0BHWmBxoHuLoUU47OP8Afil5rhCTh15JiNPCe/Gzx3GOdDMAxPYLjRguwzlyg1pF1kRFqDhTGLdRsBhWbQ0VvjTWAXHFeOMv/WbNIhUbIDI46xNQLMNgVTdclv1hb8vheq56GjnGAd58dPFu81jtgKuqRFB5ZsQRKMIFTQH9RQAC+sLIn4R68wzu+dTpCLx4aLJsWRohdAIe3BfMlanbRtnrQ+GOjaPZwnd0cTPPAkbK6onvRwpqfhRwGQQlxQSW0KczXN7LReHJjHMZIO9d17hjg4sdkNRvq4+3idzYo/P2PxeLf7OqSYRLVyAve97Wiz6ozCQmGUSG0VpPbib/ApacP7+5+us0quGBObClsG0xEsiXXv7ldJLHrW/qdnzZXa2hNa9dl7bqsPS9rz8vai7L2oqx9V9a+K2vfl7Xvy9oPZe2HsnZT1m7K2suy9rKsra+mxfXVvFg8Wc+erIsr69mVdfFlPfuyLs6sZ2fWxZv17M26uLNmdxb3CPpJ0JfsBEFN2elpZ3yIc4lWdCe54SxOzHEMdDwgLXNoOKpuDulZcuVUDX8js7esgzkC0c0VPqXj0VibxPD5LLQQbu9CymamOqg4OqacnsWlrThmJJOebM/E8Ug4dk7FNFXFNmHBuRfiyo79VgotYad6x6Rz0H1SDsMli607g+OporlGX02npmJPZcObhJSXEzDxgKUAKjYuEUeqhnwUHmVwQQg4V9En7i2ykimOz5xbYSUkAUygbZPbE5ghzLiCUfF68TgAv1QXiC07+cvjeEikbKIvtCbXF23MysXjW3IgoGz2np2B4dyvdqbLVqRmAYTFkpqB2fDXWnzo1YOWA0K7KpynJS+B8Nr/jmDeFBZmT/QvzqXAbbNnJs+4tEivHxPbB+jidfRVm7JvjtFo0c7DI3cS/Vg+udW5R7eJaksXWaLbn4DV6SnWKafZ1tX2tOIqWq2v0bg7LQJSCcjrVnOIUxdhJafUQdZcdTZ6OLacilPZ01yVwsFicJpNWOyD1gO4NRw4vTozZ0wCKJz6retQEgEJp+LouTy09yiaED18xus/7u8/yNAyBkEvNxevsd1yuo9auq3Xv43GczneuUGKp+cpiO0FsFyH5S1hkWed5qCbBwaaqSNrTeZNnl9X9E8MGFEaOCYgfZS6CYsVfdLKI4KOkzklDk9Le9QTrSxdfPrw5u7d23d3m/v3P735+YJWK3r2bTWcntH/6cBora6v6or+FfRMEmHQjdmZhgw3TDxJ03qEypzpgnQFQ95zfgsfiiDON96Ys/1pTgPCmLStr69ymn5dSxVt+C0lfY3KTK/fsAqelnokF0YVTqxEBatxSO46Ge8kE6WDSmlgUEpGeb3CmIskyDOoxJ2F5koYlAyr3o37A1Lf606GVDY2lHIR+YpJvtOpco9c/IrHuz3jj4RtXYrZ84recKluzxIUzkLj61Mq3vU1M075pqZpUwT3qGVSHWs8cbFx3ehWaDONXbBnmcigBK4zDxqu7kab+NL0PTbAD+QRfeQJWYiG5/pUXBkZ1XKk1C412mkG7URmnqDHpgE1MV4fvN5hpWbVKZYV1a5rNykz6mT5zDfiTc5mnoKabhR1RsYxxYaiYKbLwvk9YlA2GWz6odO9tjl2TSaBs+h5ZgPoTEmww0c8zFEUzsPuO/iKIlLw6pgMEwQkiDNtIE8605go2KSSzkd614KfKvpZQGmSNC4NDkPN0/2GiwvZXX+7kWFfXkBSUYphtFzvsjFc8q7hJLswX+MNgmr4tpsOssWvwG+h8YIm/psBte20DN31ZeyHuuRsjSvQpnF9j8KpJxOPHllTBEgy4yjovr7kiwMgbjh+b+VDp3dRbgiP5+2thlR0WG5wmTvlPiTdWJlOaIyvVJL2Lyp6z12COW0pZQpRyniZBWy+WLBjkm3TPWLJF7hCCam4m5EHfBF0OyGVrnaXgjR7A5ho9SCrG6tg5eq3WnDE9gMqYNXQRVVVF4JlgoQVbbkEssg5DFTlIEyHewTMRPnUGlz/cFwEyegCY8U2wSCPD3y5Cbgbp2llizuCeDjddKVDpnsxd03ubOLrWS3haj118FlS1F/ihF90g7grgK8yHSQTECgFzDhIPTJZaBoXoNTTUEPcAYXEg7g5CCWaNDJskcsPoMAyeJl4xnocwAS9RD1fFY0FX4NFkER8NEiF8UJ9fn2sM7EwYst04yz1lm7NCUbz6NYcHsww4JXNeJX3geV1utxn/hMZqYmWsOTkZI5QMDBfumFjPKQkk/4oKjk8qxXyJdMM7xH/mr9yYg9xWUuNq/smDQQ1TxZHrh0ZlZAAL694XnU8Unrd4xDgXf6lyHN6ltqRBGUQnbBkngunoU0uibkh5N8n0GJkapLfI3h2DHMXAVdxE1dlcgaFObtKv3mki/3Z+DDGgG7/iBbQjy/H4Lk+LhEx3QV5dENEN6Yf84W64V9p5HekbFrOitQsPvGm3FBgxtY7NFDmV+ezYYUG5lG2WvwJUEsDBBQAAAAIAAAAIVwx7d/7TgIAAFQEAAATAAAAcHJvbXB0cy9wcm9wb3Nlci5tZG1UyW7bMBC9+yvmmAKG0PToW9oGaIAWKBr3kFPMUCOJMDVUScqK/r5vSNtwgV5sLrO9hXoJM5nIZMjir5s9pdDlRY9YeifMkabIk4lOekRN3khDL8iyRiiIXymyaalzntP1QkImbl0mI2seNDNEirOQDeNopE1b3LS0InqcUyaNz3GlHJrN5pfWywOTS2nGr5RNNulII6dkei7JR+YJA6UhxEzeoUroSiS/G5spreNb8GiUXC8mz5Gx5hgxSMoKBttv+/1P7HCbMFnL57EiI1zoZPzMiYAi8p/ZRcW3H1iAVeqAmoOFyRWFHYz0vKM5cbleQjymyVjsIv8fSA7ICgEJk8nDuX+Y0ISNHdC4d0FocXmgg/L8qjwfEIaWAS3vP30EdsGYE4Syxnu6OwBRzK96fCj1Dixt3X5o6Ic58jW9C3O8LVwqAObXUCQpymbWwCLvZlPwL9FlVvGiUmE8mqQF7QGwcA9PQPSE6nEsA6BWMQH7xCjyHaPQ/Y6enp9/P+4Q5n1YuKW3FXqCZgjCI8tV0KsPblGDXLWRmFEra1iZsfQ7a4+UU/AnVNZDG6b1EnoWtL1Y4TJ3UwH+Qw/s3LoWI5Hwe7U1+gZAUMqV1m0Vq9CuHYpc+qAml8GO55wRqTMYzXGhpbuHhj439AVUNxDlUfNvGhn4o7RImMBWaNUaejaxdZ2zZ78VYuBy6RGTQBuLZZS+VEvVnK3ruqrRhattXc1iM0y21SdaTF2rYrni5YcJ5mzoqatvXfuXhw158SmYk3vTQtURJeKh2fwFUEsDBBQAAAAIAAAAIVyByTccIgEAANEBAAAQAAAAcHJvbXB0cy92b3Rlci5tZEWRvU7EMBCE+zzFdjQoD3ANEtAgmhMVV5p4c7bO9kbezc8J8e5MHCSKOMl61t/M+iIzucokBc9IygtXlygWzxNjKUaLGFft6QJpcAtTETKRpOSKpzyroWI0uJRQuff0zElWitinys5zfVB8TFINGBRVRlt3aFSd+dRkas447zi4sPC31wiJDQbYg1B89NBR4c2IfTTtu+57r+vTT9ed43BrzXuaf/UaRLnJd1NZYDjFG6c7YlB2N249IXrEJYMVpVEqilAfNianSmu00JSaERUyBJnrAFJw5co9ffCEM5uMNzcYfnYjKZYjCKYUYrkSJ+VHjLgdBlKm89vL+4k+4RRB8Qb4iDBFw20cE6CxSm49KQLuvmQB9FXa9Hmbkoul734BUEsDBBQAAAAIAAAAIVw0yhD6sQEAAPICAAAVAAAAc3ViX2FnZW50cy9hY3Rvci55YW1sXVI7c9swDN71K9DLkPjOlB+xnUR79w4duvkYErZ44asEFMX/vqBU29cukkB94PcAHuD7rx9qvVvDdr09qM1GrfcdcI/z+fMz+JQy6DNGhifSAYFT8rSE6Vse2bt4hpPzuIDRcT81f253kEsKmSH7gSBFBBcJC6MFQsMuRTC6lEttrh2CzkkAjwQFcyoMOtrpD7kvRozwmRipbR7kIuMHi0eTIossAosnPXjRFxNDFK7FbEEbTgXCQCycOB2xpg8ISCSOljD2zvSgpx4RH20aYUyDt2BLdX1iLLMf4S70CIxfwvc0Z/N2c/ICjjHAyxKumtp86faHN7V/XS/aJkpW3SynsUimuFz7OvipvXdINw66R0isa1jsEMrgRazO+YYVrbEmZyQkZwUozD2aj5q0qVjJjob34HjGZ82mb5uQLPoOzhiCVjuFu3flWP3WrMad3hyU4UamxGUws7xvf6OGtl3N46TV5KINtvl/DN11Ds20IV0DoER6FIDQRTvXqO2xLstUoXV8r8YiKd7LM/KxhjDQVM52jpOTRtYRi9i+ktf3yZ3/VTyf0eq6pO1FB9/8AVBLAwQUAAAACAAAACFc0fDO1uIBAABFAwAAGAAAAHN1Yl9hZ2VudHMvcHJvcG9zZXIueWFtbG2STW/bMAyG7/4VHHppgMr5aJq23mGnHnYbVgzYzdAs2hGiD0+kmwbD/vsouU3RoadYZPDyeV/yAh5+flOr7Qo2q81OrddqddNAQm1UDO4ElyFCmkLbRe91MFcgbzSWgWN0tLgCisB7hDHFMRIm6HQIkaHb6zBgaR1jOtCoO/wMo+Y9gWgh9Cn6923ghAg2lCprOoBHIi0ql4XxegMudq3OKowpLOrqQv7eucmg8AXGwCR8QUQIDjhyMw+wwcQjEOskfc0f6OdKbxMxTNkDPonUB94IkUrFEk1nrHupd2xjgFuwjB5uC1mceJy4PeApZ2Johult0A4YnzlPjdOwZ/GTwfC5OJGInfOtYAWux1Nzd3Ov7nbXi7yTbv8y/ylKAOIlmPLUHcckXynrlcqfMvLLX5nkR6cZ6ypoj83ZS2WQumTHzN3Ad9n3rJxQ2lbkTkX9mMRRHgRfHx9/PICzkm5uSIw+Sl59nBI4zAtBU4xaI+MgZIf5UKiufDToGhhQTkhtFW5/Kcvqt2Z13Or1TnVc2UCcpm6m+fSyVKjrpfD6kWn5yl17U/2/86YsvXoLvJkTr8qNNhWAKhfd9tZheQ3IrdwDT1RJzpiE+FUt//Z2eE8x12hJWtK0YWjPOCftXfUPUEsDBBQAAAAIAAAAIVzksB3HbAEAAHEEAAAUAAAAc3ViX2FnZW50cy92b3RlLnlhbWyNkUFLAzEQhe/7KwZ6sAV3TWpb6/YgIj2IUApeelvidrobSJOYybqK+N/NbrciesklkJkvmTfvjWC926ZsxmDKpouU85TNcyD57hE1vBmPjkBqsMIJpVCtTrWCcagFga8RlGmRPHg8WnTCNw5hzLIpm4DQ+wHnix6tZVV37JhnMza5BHxthALyaAnMAVjGllkygrUo69NDaJ30GJo6zJHh8AYIiaTR4ZnwCI3eB6yDi81m1U8ZVBNif7XOWEPoLggcWuPC+K78WQZ5dPfVC1fhq5NebXwtdQWoCLNEiyPm/YeJqFD7olSCKIftYMd9V0z2SKWT1gdVOTwP3skgzGI4dPDGGJWqoHvQFjbvVmylr0H6sF6rf/t3bv/Zffv48AS7fxZkCTUvRS+P8gQghdLog6wKK3wdsvxpXp2jyz7EUcWQ02jyOpqcRZPzaHIRTd5Ek8to8jaW5CyajM6IR2fEozPi0Rnx6Iz4kNE3UEsDBBQAAAAIAAAAIVz/jJBJlgEAANQCAAAYAAAAc3ViX2FnZW50cy92b3Rlcl8wMS55YW1shVLBbtUwELz7Kxb1QCvhNHnkRap7QypnTggJochNNon1HDvY65b+PV6/9IHKgVO8q92Z2ZlcwcO3L7JuazjUh042jayPCp48YYC6AT9B030AwnXDoCkFhLo61HBNC4L1zxjpHvhNBiEkizCbJ4xgCCYTIsFgtVlvKiDvbVTw/QdoN4Jxg00j9oN3hI4iOO9Q8VbEvM2AW/CbjxjeRwi4+UBl0XlajJsBbcRKXAFXp9zoH9M4I0Fdplb9q/eJtkQ9+RO6CIdjp0CXu8BEyGwQFwa1xuE9mKlwPrSfyqlW89jsfMhiLsg8kaXYF0YYUhZEjAs7BRMPPvE1mg86k10Xez/ewefjTSWcXnF3t68bMWIcgtnIeKfg6388r8TqR7QKZlxXLVuJ7aM0JH9qks+tbjo5kDAuUkjDGfHdbjNU1W22c90o3hbuah3F2whUyUDstp3w5ayTZV6yEzM6lnTZ4u9kZiXgb7mq6OWe3/qNq7vjXp0UdG1+/5OQYit56DXQP8jwNuWMWNqvJ9Di07zwCZPO/4X4DVBLAwQUAAAACAAAACFcJIVBKRoBAADkAQAAGAAAAHN1Yl9hZ2VudHMvdm90ZXJfMDIueWFtbIVPu07EMBDs/RWLaEDCOSfKReCengohIRT54o1jJX4Qbzj4e+J7wFFR2TszOztzDY8vT1zUAipRNbwsudhK+AiEM4gKQg9lcweELuKsaJkRRFHdw01SDqG3E4JKR3krSthbGsCv1B2EheJC7YhfoLz+49ANyhvUtwXLUnler5jG1M02kg1ewvM/GQrmgsZJgkHnFK851jtuib8r4vtalQ3viFmfaF66o+OV9d20aISi2MQ5uEhpc7hdOM1OXNsFT+gpSfDBI/utccyZY1II08q/vjGDPkf62cpvb41kcBlXHvJmLMQ25ulhe5pGCU29/p36bE+nKIzoV/tq22TRYP1ovblwvgB3izZIq+MBPlegISxmyBV6NSVk31BLAwQUAAAACAAAACFcBXRVcBsBAADkAQAAGAAAAHN1Yl9hZ2VudHMvdm90ZXJfMDMueWFtbIVPy07DMBC8+ysWcQGpTp02iYTv3DkhJIQiN9k4VuIHyZrC3xOnLZQTJ3tndmZnbuHx5YmLQsBO7Cqe51yUEj484QRiD76DvNoAoQ04KYoTgsj2FdzNyiJ0ZkRQ82m9FjkcDfXgFmoDPlKIVA/4Bcq1fxyaXjmN7X3G0qq8yPesxbmZTCDjnYTnfzJkzPoWRwkarVW84FgcuCH+rogfC5VXvCFm3ExTbE6ON8Y1Y2wRsmwbJm8Dzdv1dmZbdubqxjtCR7ME5x2y3xqnnCkmeT8u/Osb0+hSpB9VejujJYPruHLNmzAf6pCmh/I8DRKqYvlb9VmfT5Ef0C32u3KV9MYNxukr5yvwEFuNtDiu8KUC9T7qPlXo1Dgj+wZQSwMEFAAAAAgAAAAhXHjo4/QaAQAA5AEAABgAAABzdWJfYWdlbnRzL3ZvdGVyXzA0LnlhbWyFT0FOwzAQvPsVi7iAVKdJ5UbCd+6cEBJCkRtvEivxOsQbCr8nblMoJ06rnZmdnbmFx5cnmascdvmulEUh872Gj8A4Qa4gNFCUG2D0I06G5wkhz5SCu2g8QuMGBBPP8iov4Oi4A1qoDYSZx5mrHr/AkP3jUHeGWrT3mUhSfTlXwmKsJzeyC6Th+Z8MmfDB4qChRe+NVBLVQTqW74blUZmilDULR5GnuT473jiqh9kiZNl2nIIfOW5PvzNvxcpVdSBG4qiBAqH4rXHOmWJyCMPCv76JFilF+rlKs3GtFnAdV5/yJiyM1Zi2h/269RrKxHjzWa2vOPRIi/1uXyZR56h31F45X4GH2bbIi+MJvlTgLsxtlyo0ZogovgFQSwMEFAAAAAgAAAAhXLTzH0wbAQAA5AEAABgAAABzdWJfYWdlbnRzL3ZvdGVyXzA1LnlhbWyFT8tOwzAQvPsrFnEBqU6dKImE79w5ISSEIjfZOFbiB8mawt8TNy2UEyd7Z2ZnZ27h8eWJi1JAIYqa5zkXlYQPTziDqMD3kNc7ILQBZ0VxRhBZVcDdoixCbyYEtWzyRuRwNDSAW6kd+EghUjPiFyjX/XFoB+U0dvcZS1J5Wa9Yh0s7m0DGOwnP/2TImPUdThI0Wqt4ybE8cEP8XRE/liqveUvMuIXm2G6ON8a1U+wQsmwfZm8DLfvT7cx27Mw1rXeEjhYJzjtkvzW2nCkmeT+t/Osb0+hSpJ+t9PZGSwbXceUpb8J8aEKaHqrzNEqoy/Vv1WdzPkV+RLfaF1WdRINxo3H6yvkKPMROI62OJ/hSgQYf9ZAq9GpakH0DUEsDBBQAAAAIAAAAIVxMM4K/GgEAAOQBAAAYAAAAc3ViX2FnZW50cy92b3Rlcl8wNi55YW1shU/LTsMwELz7KxZxAalOnSq1hO/cOSEkhCI32ThW4gfJmsLfE7cplBMne2dmZ2du4fHliYtKwE7sJC9LLvYKPgLhBEJC6KCUGyB0ESdNaUIQhRRwN2uH0NkRQc9neS1KOFrqwS/UBkKimKge8Au0b/84NL32Btv7gmWpuqxL1uLcTDaSDV7B8z8ZCuZCi6MCg85pXnGsDtwSf9fEj5UuJW+IWT/TlJqz4431zZhahKLYxim4SPP2dLtwLVu5ugme0NOswAeP7LfGOWeOSSGMC//6xgz6HOlnK7+dNYrBdVx1ypuxEOuYp4f9Og0KZLX8nf6s11MUBvSL/W4vs6i3frDeXDlfgYfUGqTF8QRfKlAfkulzhU6PM7JvUEsDBBQAAAAIAAAAIVyt472OGwEAAOQBAAAYAAAAc3ViX2FnZW50cy92b3Rlcl8wNy55YW1shU/LTsMwELz7KxZxAQmnTpWm4Dt3TggJociNN46V2A7xmsLfE/cB5cTJ3pnZ2ZlreHx54qISsBbrmpclFxsJH4FwBrGF0EFZ3wGhm3BWlGYEUdT3cBOVQ+jsiKDiUd6IEvaWevALdQch0ZSoGfALlNd/HNpeeYP6tmBZKs/rW6YxtrOdyAYv4fmfDAVzQeMowaBzilccqx23xN8V8X2lypq3xKyPNKf26HhlfTsmjVAUq2kObqK4OtwunGYnrmmDJ/QUJfjgkf3WOObMMSmEceFf35hBnyP9bOW3s0YyuIwrD3kzFqZmytPD5jQNEupq+Tv12ZxOURjQL/brTZ1FvfWD9ebC+QLcJW2QFscDfK5AfUimzxU6NUZk31BLAwQUAAAACAAAACFcQkvEaBsBAADkAQAAGAAAAHN1Yl9hZ2VudHMvdm90ZXJfMDgueWFtbIVPQU7DMBC8+xWLuIBUp0mVpsV37pwQEkKRG28cK/HaJBsKvydJWygnTvbO7MzO3MLjy5NM8xQ26aaQWSbTrYKPwNhDuodQQ1asgNFH7DWPPUKa7Aq4G7RHqF2HoIfTeplmcHTcAE3UCsLIceSyxS/QZP44VI0mi+Y+EfOqusj3wuBQ9S6yC6Tg+Z8MifDBYKfAovda5hLzg3Qs3zXLY66zQlYsHA3cj9XJ8cZR1Y0GIUnWsQ8+8rBebifeiDNXVoEYiQcFFAjFb41Tzjkmh9BN/OubsEhzpB/V/NbOKgHXcdWSd8ZCLOM8PWzPU6ugyKe/15/l+RSHFmmy32wXSeOodWSvnK/Aw2gs8uS4wJcK3ITRNnOFWncDim9QSwMEFAAAAAgAAAAhXPs3Sb0aAQAA5AEAABgAAABzdWJfYWdlbnRzL3ZvdGVyXzA5LnlhbWyFT8tOwzAQvPsrFnEBqU6dKo2o79w5ISSEIjfZOFbiB/Gawt8T9wHlxGm1M7OzM7fw+PLERSVgIzY1L0suthI+POEMYge+h7JeAaENOCtKM4IoHiq4i8oi9GZCUPEkb0QJB0MDuIVagU8UEjUjfoFy3R+HdlBOY3dfsCyVl/Md6zC2swlkvJPw/E+Gglnf4SRBo7WKVxyrPTfE3xXxQ6XKmrfEjIs0p/bkeGNcO6UOoSjWYfY2UFwffxe2Y2euab0jdBQlOO+Q/dY45cwxyftp4V/fmEaXI/1c5dkbLRlcx5XHvBnzoQl5223P2yihzoxVn835FfkR3WK/2dZZNBg3GqevnK/Afeo00uJ4hC8VaPBJD7lCr6aI7BtQSwMEFAAAAAgAAAAhXAN43zwbAQAA5AEAABgAAABzdWJfYWdlbnRzL3ZvdGVyXzEwLnlhbWyFT8tOwzAQvPsrFnEBqU7tKI3U3LlzQkgIRW68cazED5I1hb8nSQOUEyfvzoxnZ27h4fmRi0JALvKSS8nFoYL3QDiCFBBakOUOCF3EUVEaEUR2zOFuUg6htQOCmi7yWkg4W+rAz9QOQqKYqO7xE5TXfxyaTnmD+j5ji3S7VkvBNE7NaCPZ4Ct4+idDxlzQOFRg0DnFC47FiVvib4r4uVCy5A0x6ycaU3NxvLG+GZJGyLJ9HIOLNO3X25nTbOPqJnhCT1MFPnhkvzUuOZeYFMIw8y+vzKBfIv38Wt7WmorBddxqzbtgIdZx3Q7b1ldQFvPs1Ee9naLQo5/t80O5iDrre+vNlfMVeEraIM2OK/xdgbqQTLdUaNUwIfsCUEsDBBQAAAAIAAAAIVxBZz5JHAEAAOQBAAAYAAAAc3ViX2FnZW50cy92b3Rlcl8xMS55YW1shU/LTsMwELz7KxZxAal27SqNVN+5c0JICEVuskmsxOuQrCn8PUkboJw42Z4Zz+MWHp4fpc407PQul8ZIvbfwHhlHMAZiDSbfAGMYcHScRgSjtIa7yQWE2vcIbrrIC23g5LkFmqkNxMRD4qLDT3BU/XEoW0cNVvdKLNI1rTBGVDiVox/YR7Lw9E8HJUKssLfQYAhOZhKzo/Qs3xzLU+ZMLksWniYeU3lxvPFU9qlCUGo7jDEMPG3P2SpUYuWKMhIj8WSBIqH4nXHpudTkGPuZf3kVDdJS6efXcta+sQKu69pz3wWLQzFY0OqwX1+dhTyb78F9FGsUxw5ptt/t80XUeuo8NVfOV+AxVQ3y7HiGvydwG1PTLhNq108ovgBQSwMEFAAAAAgAAAAhXPnF3gIcAQAA5AEAABgAAABzdWJfYWdlbnRzL3ZvdGVyXzEyLnlhbWyFT01PxCAUvPMrnvGiydKFptsod++ejIkxDVteKWmB2r66+u8t3arryRMwM8zHNTw8P3JRCMhFXnIpuTgoeI+EI8gcYgOy3AGhH3DUNI8IMhN3cDNpj9C4HkFPZ3klJJwctRAWagdxpmGmqsNP0MH8cahbHSya24wl6ZZWyZwZnOrRDeRiUPD0T4eM+WiwV2DRe80LjsWRO+Jvmvip0LLkNTEXJhrn+ux45ULdzwYhy/bDGP1A037NzrxhG1fVMRAGmhSEGJD9zjj3TDUpxn7hX16ZxZAq/fxKZ+OsYnBZV619ExaHalAgsvvD9uoUlMVy9/qj2qIodhgW+/xQJlHrQueCvXC+AI+zsUiL4wp/T6A2zrZNExrdT8i+AFBLAwQUAAAACAAAACFc2DTKWxwBAADkAQAAGAAAAHN1Yl9hZ2VudHMvdm90ZXJfMTMueWFtbIVPTU+EMBC991eM8aLJlqW7LMn27t2TMTGGdOkADXSKMHX13wss6nry1Ol7M+/jFh6eH2WapbBLd7lUSqYHDe+BcQC1h1CByjfA6HscDMcBQSUqh7vReITKdQhmvKwXqYKz4wZoojYQIveRixY/wZD9o1A2hmq094mYV1e3Qu2FxbEcXM8ukIanfzIkwgeLnYYavTcyk5idpGP5ZlieM6NyWbJwNPIQy4vijaOyixYhSbb9EHzP43bxTrwVK1eUgRiJRw0UCMVvjUvOOSaH0E38y6uokeZIP1fzW7laC7iOq5e8Mxb6oteQJsfD+ms15Nk0e/NRrFYcWqRJfndYThpHraP6SvkKPEVbI0+KC/xdgZsQ62auUJluRPEFUEsDBBQAAAAIAAAAIVySEOldGgEAAOQBAAAYAAAAc3ViX2FnZW50cy92b3Rlcl8xNC55YW1shU9NT4QwEL33V4zxoontUsKS2Lt3T8bEGNKFoTTQKcLg6r+X7qKuJ0/NvPf6Pq7h4flRZkUGeZaXUmuZ7Q28R8YJdAGxBV3eAWMYcbK8TAha5QXczDYgtH5AsPNZXmUajp47oJW6g7jwuHDV4ydYav441J0lh82tEkm6pVW6EA3O9eRH9pEMPP3TQYkQGxwMOAzBykJicZCe5ZtleSysLmXNwtPM01KfHa881cPSICi1G6cYRp53p2wVGrFxVR2JkXg2QJFQ/M4490w1OcZh5V9ehUNKlX5+pbf1zgi4rGtOfRMWx2o0kKn7/Xb1BsrEBPtRbVEce6TVPt+XSdR56j25C+cL8LA0Dnl1PMHfE7iLi+vShNYOM4ovUEsDBBQAAAAIAAAAIVxeCxXlHAEAAOQBAAAYAAAAc3ViX2FnZW50cy92b3Rlcl8xNS55YW1shU9NT4QwFLz3VzzjRZNtlyKQ2Lt3T8bEGNKFR2mgrwgPV/+9sIu6njy1nZnOxzU8PD/KJEsgTdJCai2T3MB7ZBxB5xAb0MUOGMOAo+V5RNDqLoWbyQaExvcIdjrLy0TD0XMLtFA7iDMPM5cdfoKl+o9D1VpyWN8qsUq3tFLnosapGv3APpKBp386KBFijb0BhyFYmUnMDtKzfLMsj5nVhaxYeJp4nKuz45Wnqp9rBKX2wxjDwNP+lK1CLTaurCIxEk8GKBKK3xnnnmtNjrFf+JdX4ZDWSj+/1rPxzgi4rGtOfVcsDuVgIFH3+fbqDBTZcg/2o9yiOHZIi32aF6uo9dR5chfOF+Bhrh3y4niCvydwG2fXrhMa208ovgBQSwMEFAAAAAgAAAAhXCJC3/QlAQAA8QEAABgAAABzdWJfYWdlbnRzL3ZvdGVyXzE2LnlhbWyFT8tOwzAQvPsrFnEBqU6TKo2EOXPnhJAQitx4Y1uJ7RBvKPw9dhugnLj4MTM7M3sND8+PvKxL2JW7hlcVL/cC3gPhDFUDoU/nBgjdhLOkZUaoiiS+IYNgrDYY6R6idAi9HRFkPM+2ZQVHSwZ8ojYQFpoWagf8BOnVH7vOSK9R3RYsS9fotmqYwtjNdiIbvICnfwoVzAWFowCNzklec6wP3BJ/k8SPtawa3hGzPtK8dGfHK+u7cVEIRbGd5uAmittTduEUW7m2C57QUxTgg0f2u8a5Z65JIYyJf3llGn2u9DOV795qweCyrjj1zViY2klAWdzt198goKnT28mPdo2iMKBP9rt9k0XG+sF6feF8AR4WpZGS4wn+XoFMWLTJK/RyjMi+AFBLAQIUABQAAAAIAAAAIVzWj3oRmgEAANICAAAKAAAAAAAAAAAAAACkAQAAAABhZ2VudC55YW1sUEsBAhQAFAAAAAgAAAAhXPz0ZZVsAAAAgQAAABUAAAAAAAAAAAAAAKQBwgEAAGNvbmZpZ3Mvc2FtcGxpbmcueWFtbFBLAQIUABQAAAAIAAAAIVynzIDmhQEAAHsCAAAeAAAAAAAAAAAAAACkAWECAABjb25maWdzL3NhbXBsaW5nX3Byb3Bvc2VyLnlhbWxQSwECFAAUAAAACAAAACFcmZgbxBEBAACUAQAAEAAAAAAAAAAAAAAApAEiBAAAZXZhbF9jb25maWcueWFtbFBLAQIUABQAAAAIAAAAIVyLolMBjQgAAJwSAAAQAAAAAAAAAAAAAACkAWEFAABwcm9tcHRzL2FjdG9yLm1kUEsBAhQAFAAAAAgAAAAhXDHt3/tOAgAAVAQAABMAAAAAAAAAAAAAAKQBHA4AAHByb21wdHMvcHJvcG9zZXIubWRQSwECFAAUAAAACAAAACFcgck3HCIBAADRAQAAEAAAAAAAAAAAAAAApAGbEAAAcHJvbXB0cy92b3Rlci5tZFBLAQIUABQAAAAIAAAAIVw0yhD6sQEAAPICAAAVAAAAAAAAAAAAAACkAesRAABzdWJfYWdlbnRzL2FjdG9yLnlhbWxQSwECFAAUAAAACAAAACFc0fDO1uIBAABFAwAAGAAAAAAAAAAAAAAApAHPEwAAc3ViX2FnZW50cy9wcm9wb3Nlci55YW1sUEsBAhQAFAAAAAgAAAAhXOSwHcdsAQAAcQQAABQAAAAAAAAAAAAAAKQB5xUAAHN1Yl9hZ2VudHMvdm90ZS55YW1sUEsBAhQAFAAAAAgAAAAhXP+MkEmWAQAA1AIAABgAAAAAAAAAAAAAAKQBhRcAAHN1Yl9hZ2VudHMvdm90ZXJfMDEueWFtbFBLAQIUABQAAAAIAAAAIVwkhUEpGgEAAOQBAAAYAAAAAAAAAAAAAACkAVEZAABzdWJfYWdlbnRzL3ZvdGVyXzAyLnlhbWxQSwECFAAUAAAACAAAACFcBXRVcBsBAADkAQAAGAAAAAAAAAAAAAAApAGhGgAAc3ViX2FnZW50cy92b3Rlcl8wMy55YW1sUEsBAhQAFAAAAAgAAAAhXHjo4/QaAQAA5AEAABgAAAAAAAAAAAAAAKQB8hsAAHN1Yl9hZ2VudHMvdm90ZXJfMDQueWFtbFBLAQIUABQAAAAIAAAAIVy08x9MGwEAAOQBAAAYAAAAAAAAAAAAAACkAUIdAABzdWJfYWdlbnRzL3ZvdGVyXzA1LnlhbWxQSwECFAAUAAAACAAAACFcTDOCvxoBAADkAQAAGAAAAAAAAAAAAAAApAGTHgAAc3ViX2FnZW50cy92b3Rlcl8wNi55YW1sUEsBAhQAFAAAAAgAAAAhXK3jvY4bAQAA5AEAABgAAAAAAAAAAAAAAKQB4x8AAHN1Yl9hZ2VudHMvdm90ZXJfMDcueWFtbFBLAQIUABQAAAAIAAAAIVxCS8RoGwEAAOQBAAAYAAAAAAAAAAAAAACkATQhAABzdWJfYWdlbnRzL3ZvdGVyXzA4LnlhbWxQSwECFAAUAAAACAAAACFc+zdJvRoBAADkAQAAGAAAAAAAAAAAAAAApAGFIgAAc3ViX2FnZW50cy92b3Rlcl8wOS55YW1sUEsBAhQAFAAAAAgAAAAhXAN43zwbAQAA5AEAABgAAAAAAAAAAAAAAKQB1SMAAHN1Yl9hZ2VudHMvdm90ZXJfMTAueWFtbFBLAQIUABQAAAAIAAAAIVxBZz5JHAEAAOQBAAAYAAAAAAAAAAAAAACkASYlAABzdWJfYWdlbnRzL3ZvdGVyXzExLnlhbWxQSwECFAAUAAAACAAAACFc+cXeAhwBAADkAQAAGAAAAAAAAAAAAAAApAF4JgAAc3ViX2FnZW50cy92b3Rlcl8xMi55YW1sUEsBAhQAFAAAAAgAAAAhXNg0ylscAQAA5AEAABgAAAAAAAAAAAAAAKQByicAAHN1Yl9hZ2VudHMvdm90ZXJfMTMueWFtbFBLAQIUABQAAAAIAAAAIVySEOldGgEAAOQBAAAYAAAAAAAAAAAAAACkARwpAABzdWJfYWdlbnRzL3ZvdGVyXzE0LnlhbWxQSwECFAAUAAAACAAAACFcXgsV5RwBAADkAQAAGAAAAAAAAAAAAAAApAFsKgAAc3ViX2FnZW50cy92b3Rlcl8xNS55YW1sUEsBAhQAFAAAAAgAAAAhXCJC3/QlAQAA8QEAABgAAAAAAAAAAAAAAKQBvisAAHN1Yl9hZ2VudHMvdm90ZXJfMTYueWFtbFBLBQYAAAAAGgAaAO0GAAAZLQAAAAA="               # filled in by tools/build_notebook.py --embed
# Model: taken from the embedded zip's agent.yaml. Set MODEL_NAME only as a cross-check (the run asserts the
# zip declares exactly this name), e.g. --set MODEL_NAME='"gemma-4-e2b-it-qat-w4a16-ct"'. The kernel-metadata
# model_sources must attach google/gemma-4/other/<name>/2 for the same name.
MODEL_NAME = "gemma-4-e4b-it-qat-w4a16-ct"   # overridden by --set

# Compaction settings: Q7 ANSWERED [V] from HARNESS_README.md section 7.2 (interval 15, overlap 2,
# token_threshold 14,336, event_retention_size 5). The 'interval 5' reading confused retention with interval.
COMPACTION_INTERVAL = 15   # [V] HARNESS_README §7.2: interval 15, overlap 2, threshold 14,336, retention 5
COMPACTION_TOKEN_THRESHOLD = 14336

In [ ]:
# ---------------------------------------------------------------- environment + wheelhouse (hosts' recipe)
# Adapted from the organizers' Getting-Started notebook as mirrored by community repos.  # VERIFY
import glob, importlib, json, os, shutil, subprocess, sys, time, zipfile, io, base64
from pathlib import Path

os.environ.update({
    'LITELLM_LOCAL_MODEL_COST_MAP': 'True', 'TRANSFORMERS_NO_TF': '1', 'VLLM_WORKER_MULTIPROC_METHOD': 'spawn',
    'VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS': '1', 'VLLM_ENGINE_READY_TIMEOUT_S': '1200',
    'VLLM_NO_USAGE_STATS': '1', 'OTEL_SDK_DISABLED': 'true', 'PYTORCH_CUDA_ALLOC_CONF': 'expandable_segments:True',
})
WHEELHOUSE_DIR = next(Path(p) for p in ['/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse',
                                        '/kaggle/input/gemma-4-developer-agent-wheelhouse'] if Path(p).exists())
for pat in ('/usr/local/lib/python*/dist-packages/*cutlass*.pth', '/usr/local/lib/python*/site-packages/*cutlass*.pth'):
    for pth in glob.glob(pat):
        try:
            os.unlink(pth)
        except OSError:
            pass
tmp_whl = Path('/tmp/wheelhouse'); tmp_whl.mkdir(parents=True, exist_ok=True)
for w in WHEELHOUSE_DIR.glob('*.whl'):
    if 'cutlass' in w.name.lower():
        continue
    name = w.name.replace('cu128', '+cu128') if ('cu128' in w.name and '+' not in w.name) else w.name
    if not (tmp_whl / name).exists():
        os.symlink(w, tmp_whl / name)
# The wheelhouse ships cp312 binary wheels (orchestrator env = Python 3.12), but the grading sandbox and
# Kaggle's CPU image are Python 3.13 (docker/Dockerfile.sandbox is `python:3.13-slim`). A single bulk
# `pip install ... check=True` aborts the WHOLE install when one wheel is incompatible, which is what killed
# the first EXP-000 push (apache_tvm_ffi cp312 on py3.13). Filter by interpreter tag, then assert that the
# packages we actually need for this run arrived.
# Import names, not distribution names: the `google_adk` wheel installs the `google.adk`
# namespace package, so importing 'google_adk' always fails. [V] google_adk-1.36.1.dist-info
REQUIRED_PKGS = ('swegemma', 'adk_submission', 'adk_eval_core', 'google.adk')

def _wheel_is_compatible(path: Path) -> bool:
    """True when the wheel's tags intersect this interpreter's supported tags."""
    try:
        from packaging.tags import sys_tags
        from packaging.utils import parse_wheel_filename
    except Exception:  # noqa: BLE001 - fall back to accepting everything
        return True
    try:
        _, _, _, tags = parse_wheel_filename(path.name)
    except Exception:  # noqa: BLE001 - odd filename: let pip decide
        return True
    supported = set(sys_tags())
    return any(t in supported for t in tags)

all_wheels = sorted(tmp_whl.glob('*.whl'))
usable = [w for w in all_wheels if _wheel_is_compatible(w)]
skipped = [w.name for w in all_wheels if w not in usable]
T_INSTALL = time.time()
proc = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--force-reinstall',
                       *(str(w) for w in usable)], capture_output=True, text=True)
if proc.returncode != 0:
    print('bulk install failed; retrying wheel-by-wheel to isolate the bad ones')
    failed = []
    for w in usable:
        r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--force-reinstall', str(w)],
                           capture_output=True, text=True)
        if r.returncode != 0:
            failed.append(w.name)
    skipped += failed
importlib.invalidate_caches()
print(f'wheelhouse: installed {len(all_wheels) - len(skipped)}/{len(all_wheels)} wheels in {time.time() - T_INSTALL:.0f}s '
      f'(python {sys.version_info.major}.{sys.version_info.minor})')
if skipped:
    print(f'  incompatible/failed ({len(skipped)}):', ', '.join(sorted(skipped)[:12]))
missing = []
for _pkg in REQUIRED_PKGS:
    try:
        importlib.import_module(_pkg)
    except Exception as _exc:  # noqa: BLE001
        missing.append(f'{_pkg} ({_exc.__class__.__name__}: {_exc})')
assert not missing, 'required packages unavailable: ' + '; '.join(missing)
print('  required packages present:', ', '.join(REQUIRED_PKGS))

DATA_DIR = next(Path(p) for p in ['/kaggle/input/competitions/gemma-4-developer-agent',
                                  '/kaggle/input/gemma-4-developer-agent'] if Path(p).exists())
RUN_DIR = Path('/kaggle/working/runs') / EXP_ID
for sub in ('traces', 'patches', 'logs', 'phase0'):
    (RUN_DIR / sub).mkdir(parents=True, exist_ok=True)
print('data:', DATA_DIR, ' run dir:', RUN_DIR)

In [ ]:
# ---------------------------------------------------------------- Phase 0 checks (plan §3.4)
# Answers Q1/Q2/Q3/Q7/Q9 by reading the installed harness source; saves HARNESS_README.md and
# sample_submission/ to the run dir so they can be downloaded and read offline.
if RUN_PHASE0_CHECKS:
    import importlib.metadata as md, re
    report = {'python': sys.version, 'packages': {}}
    for pkg in ('swegemma', 'adk-submission', 'adk-eval-core', 'google-adk', 'vllm', 'litellm'):
        try:
            report['packages'][pkg] = md.version(pkg)
        except md.PackageNotFoundError:
            report['packages'][pkg] = None
    for name in ('HARNESS_README.md',):
        src = DATA_DIR / name
        if src.exists():
            shutil.copy(src, RUN_DIR / 'phase0' / name)
    ss = DATA_DIR / 'sample_submission'
    if ss.exists():
        shutil.copytree(ss, RUN_DIR / 'phase0' / 'sample_submission', dirs_exist_ok=True)
        print('--- sample_submission files (Q2) ---')
        for p in sorted(ss.rglob('*')):
            if p.is_file() and p.suffix in ('.yaml', '.yml', '.md', '.txt', '.json'):
                print(f'===== {p.relative_to(ss)}'); print(p.read_text()[:4000])

    def grep_pkg(pkg: str, patterns: list[str]) -> dict:
        hits = {}
        try:
            root = Path(importlib.import_module(pkg).__file__).parent
        except Exception as exc:  # noqa: BLE001
            return {'error': str(exc)}
        for pat in patterns:
            rx = re.compile(pat)
            for f in root.rglob('*.py'):
                for i, line in enumerate(f.read_text(errors='ignore').splitlines(), 1):
                    if rx.search(line):
                        hits.setdefault(pat, []).append(f'{f.relative_to(root)}:{i}: {line.strip()[:160]}')
        return hits

    report['Q1_tool_registry'] = grep_pkg('swegemma', [r'register\(', r'def (run_command|read_file|edit_file|write_file|get_status|submit_patch|get_code_neighbors|search_similar_code|get_code_subgraph)\b'])
    report['Q1_skill_tools'] = grep_pkg('adk_submission', [r'run_skill_script|load_skill_resource|SkillToolset'])
    report['Q2_eval_config_keys'] = grep_pkg('swegemma', [r'max_time_minutes|max_tool_calls|max_turns|timeout_seconds', r'load_submission_eval_config'])
    report['Q2_agent_tool_schema'] = grep_pkg('adk_submission', [r'agent_tool|config_path|skip_summarization'])
    report['Q3_thinking_forwarding'] = grep_pkg('adk_submission', [r'thinking_budget|thinking_token_budget|enable_thinking|include_thoughts'])
    report['Q7_compaction'] = grep_pkg('swegemma', [r'compaction_interval|token_threshold|EventsCompactionConfig'])
    report['Q9_extensions_limits'] = grep_pkg('swegemma', [r'ALLOWED_SUBMISSION_EXTENSIONS|MAX_SUBMISSION_SIZE_BYTES|allowed_file_extensions|max_total_size_bytes'])
    report['Q2_grading_timeout'] = grep_pkg('swegemma', [r'timeout', ]) if False else grep_pkg('swegemma', [r'verification.*timeout|timeout.*verif|command_timeout_seconds'])
    (RUN_DIR / 'phase0' / 'phase0_report.json').write_text(json.dumps(report, indent=1))
    print(json.dumps({k: (v if k == 'packages' else {kk: len(vv) for kk, vv in v.items()} if isinstance(v, dict) else v)
                      for k, v in report.items()}, indent=1))
    print('full grep hits saved to', RUN_DIR / 'phase0' / 'phase0_report.json')

In [ ]:
# ---------------------------------------------------------------- tasks + candidates
from swegemma.models import load_tasks  # [V] swegemma/models/__init__.py exports load_tasks(path)->list[Task]
TASKS_PATH = DATA_DIR / 'tasks.jsonl'
all_tasks = load_tasks(TASKS_PATH)
from collections import Counter
print('repo counts:', Counter(t.repo for t in all_tasks))
if TASK_IDS:
    want = set(TASK_IDS)
    candidates = [t for t in all_tasks if t.instance_id in want]
else:
    pool = [t for t in all_tasks if (not REPO_FILTER or t.repo == REPO_FILTER) and t.instance_id not in set(EXCLUDE_IDS)]
    candidates = sorted(pool, key=lambda t: t.instance_id)[:N_TASKS]
print(f'{len(all_tasks)} tasks; candidates ({len(candidates)}):', [t.instance_id for t in candidates])

In [ ]:
# ---------------------------------------------------------------- unpack our submission
AGENT_DIR = Path('/kaggle/working/agent')
shutil.rmtree(AGENT_DIR, ignore_errors=True); AGENT_DIR.mkdir(parents=True)
if SUBMISSION_B64:
    zipfile.ZipFile(io.BytesIO(base64.b64decode(SUBMISSION_B64))).extractall(AGENT_DIR)
elif Path(SUBMISSION_ZIP).exists():
    zipfile.ZipFile(SUBMISSION_ZIP).extractall(AGENT_DIR)
elif not RUN_AGENT and (DATA_DIR / 'sample_submission').is_dir():
    # A gold-control run never calls a model, so it needs no submission of ours. Fall back to the
    # official sample_submission: it is a real, schema-valid submission dir, so the model/adapter/limits
    # validation below still runs for real instead of being skipped.
    shutil.copytree(DATA_DIR / 'sample_submission', AGENT_DIR, dirs_exist_ok=True)
    print('no submission supplied; using official sample_submission (RUN_AGENT=False)')
else:
    raise SystemExit('no submission: set SUBMISSION_ZIP to your uploaded dataset file or embed with tools/build_notebook.py')
print('agent files:', sorted(str(p.relative_to(AGENT_DIR)) for p in AGENT_DIR.rglob('*') if p.is_file()))
import yaml
EVAL = (yaml.safe_load((AGENT_DIR / 'eval_config.yaml').read_text()) or {}).get('evaluation', {}) \
    if (AGENT_DIR / 'eval_config.yaml').exists() else {}
if not RUN_AGENT:
    # The official sample eval_config is deliberately tiny (1 min / 10 tool calls) to demo the schema.
    # Gold control runs no agent loop, so those caps only risk failing a sound task for budget reasons
    # and corrupting the Q5 answer. Give the grader room; the agent budget is irrelevant here.
    EVAL = {**EVAL, 'timeout_seconds': 300, 'max_time_minutes': 60.0, 'max_tool_calls': 100}
print('eval_config:', EVAL)

In [ ]:
# ---------------------------------------------------------------- vLLM server (hosts' recipe)  # VERIFY
# CPU-safe imports only: a gold-control run (RUN_AGENT=False) never loads vLLM/litellm/torch, and on
# Kaggle's Python 3.13 CPU image those cp312 wheels are not installed at all.
from adk_submission import ModelRegistry, discover_adapters, discover_declared_models
from swegemma.config import ALLOWED_ADAPTER_EXTENSIONS, EvalConfig, build_submission_limits

# D-004 2026-11-05: TARGET was hard-coded to 31B; it now comes from the zip, so a small-variant candidate
# runs without editing this file. Mirror of tools/submission_rules.py ALLOWED_MODEL_NAMES (the notebook runs
# on Kaggle and cannot import tools/). 12B is excluded there and here: it needs vLLM --hf-overrides
# (HARNESS_README §3.1 MODEL_VLLM_EXTRA_ARGS), which this cell does not pass.
ALLOWED_MODELS = ('gemma-4-31b-it-qat-w4a16-ct', 'gemma-4-e4b-it-qat-w4a16-ct', 'gemma-4-e2b-it-qat-w4a16-ct')
# [V] adk_submission.discovery.discover_declared_models(submission_dir) -> set[str]; walks the root
# agent, sub_agents and agent_tools. There is no `validate_single_declared_model` anywhere in the
# wheelhouse -- that was an invented name. Exactly one model may be declared, so assert the set size.
declared = discover_declared_models(str(AGENT_DIR))
assert len(declared) == 1, f'submission must declare exactly one model, got {sorted(declared)}'
TARGET = next(iter(declared)).strip()
for _prefix in ('openai/', 'google/', 'hosted_vllm/', 'custom/'):   # [V] HARNESS_README §3.2 normalize_model_name
    if TARGET.startswith(_prefix):
        TARGET = TARGET[len(_prefix):]
        break
assert TARGET in ALLOWED_MODELS, f'declared model {TARGET!r} not in {ALLOWED_MODELS}'
assert not MODEL_NAME or MODEL_NAME == TARGET, f'MODEL_NAME={MODEL_NAME!r} but the zip declares {TARGET!r}'
MODEL_DIR = f'/kaggle/input/models/google/gemma-4/other/{TARGET}/2'   # [V] path layout EXP-006 resolved 'exact'
print('model:', TARGET, '->', MODEL_DIR)
adapters = discover_adapters(str(AGENT_DIR), adapter_extensions=ALLOWED_ADAPTER_EXTENSIONS)
# EXP-000 gold control needs no model: it applies each task's reference patch and runs test_patch.
# Gate the server so a gold-control-only run costs ZERO GPU quota (plan §5.7) and can use a CPU notebook.
NEED_SERVER = RUN_AGENT or MEASURE_DECODE_SPEED
server = None
SERVER_START_S = None
target_id = None
models = ModelRegistry()

if NEED_SERVER:
    import litellm, torch
    from adk_submission import VllmConfig, VllmServer
    from google.adk.models.lite_llm import LiteLlm
    try:
        from adk_submission.resolvers.generation import install_litellm_reasoning_patch
        install_litellm_reasoning_patch()
    except Exception as exc:  # noqa: BLE001
        print('reasoning patch not installed:', exc)
    litellm.drop_params = True
    gpu_count = torch.cuda.device_count() if torch.cuda.is_available() else 0
    assert gpu_count >= 1, 'no GPU: select the L4 x4 accelerator'
    assert Path(MODEL_DIR, 'config.json').exists(), (
        f'{MODEL_DIR} missing: kernel-metadata.json model_sources must list google/gemma-4/other/{TARGET}/2')
    tp = 4 if gpu_count >= 4 else (2 if gpu_count >= 2 else 1)
    t0 = time.time()
    server = VllmServer(VllmConfig(
        model=MODEL_DIR, port=8000, host='127.0.0.1', tool_call_parser='gemma4', reasoning_parser='gemma4',
        default_chat_template_kwargs={'enable_thinking': True}, max_model_len=32768,
        dtype='bfloat16' if torch.cuda.is_bf16_supported() else 'auto', gpu_memory_utilization=0.90,
        enable_auto_tool_choice=True, enable_lora=True, max_loras=8, max_lora_rank=128,
        tensor_parallel_size=tp, startup_timeout=60 * 20), adapter_manifest=adapters)
    server.start()
    SERVER_START_S = round(time.time() - t0, 1)
    print(f'vLLM up in {SERVER_START_S}s (tp={tp}, gpus={gpu_count}) at {server.base_url}')
    target_id = getattr(server, 'effective_model_path', MODEL_DIR)
    for alias in {next(iter(declared)), TARGET}:
        models.register(alias, LiteLlm(model=f'openai/{target_id}', api_base=server.base_url, api_key='EMPTY'))
else:
    print('vLLM server SKIPPED (RUN_AGENT and MEASURE_DECODE_SPEED are both False) - no GPU required.')

In [ ]:
# ---------------------------------------------------------------- Q10: decode speed probe
DECODE = None
if MEASURE_DECODE_SPEED:
    import urllib.request
    body = json.dumps({'model': target_id, 'messages': [{'role': 'user', 'content': 'Count from 1 to 400 separated by spaces.'}],
                       'max_tokens': 400, 'temperature': 0.0, 'chat_template_kwargs': {'enable_thinking': False}}).encode()
    req = urllib.request.Request(server.base_url.rstrip('/') + '/chat/completions', data=body,
                                 headers={'Content-Type': 'application/json'}, method='POST')
    t0 = time.time()
    with urllib.request.urlopen(req, timeout=600) as resp:
        out = json.loads(resp.read())
    dt = time.time() - t0
    ct = out.get('usage', {}).get('completion_tokens', 0)
    DECODE = {'completion_tokens': ct, 'seconds': round(dt, 2), 'tok_s': round(ct / dt, 1) if dt else None}
    print('decode probe:', DECODE)

In [ ]:
# ---------------------------------------------------------------- evaluator helpers
import asyncio, concurrent.futures
from google.adk.agents.context_cache_config import ContextCacheConfig
from google.adk.apps._configs import EventsCompactionConfig  # [V] google/adk/apps/_configs.py:50
from swegemma.evaluate import Evaluator

def run_sync(fn):
    try:
        loop = asyncio.get_running_loop()
    except RuntimeError:
        loop = None
    if loop is not None and loop.is_running():
        with concurrent.futures.ThreadPoolExecutor(max_workers=1) as ex:
            return ex.submit(lambda: asyncio.run(fn())).result()
    return asyncio.run(fn())

def make_config(results_dir: Path) -> EvalConfig:
    limits, gen = build_submission_limits()
    return EvalConfig(
        tasks_path=TASKS_PATH, snapshots_dir=DATA_DIR / 'snapshots', results_dir=results_dir,
        submission_dir=AGENT_DIR, models=models, sandbox='subprocess',
        timeout_seconds=int(EVAL.get('timeout_seconds', 300)),
        max_time_minutes=float(EVAL.get('max_time_minutes', 60.0)),
        max_tool_calls=int(EVAL.get('max_tool_calls', 100)),
        max_turns=int(EVAL['max_turns']) if EVAL.get('max_turns') is not None else None,
        limits=limits, generation_constraints=gen, adapter_manifest=adapters,
        context_cache_config=ContextCacheConfig(min_tokens=2048, ttl_seconds=1800, cache_intervals=10),
        events_compaction_config=EventsCompactionConfig(compaction_interval=COMPACTION_INTERVAL, overlap_size=2,
                                                        token_threshold=COMPACTION_TOKEN_THRESHOLD, event_retention_size=5),
        graph_dir=str(DATA_DIR / 'graphs'), embeddings_dir=str(DATA_DIR / 'embeddings'),
        wheels_dir=DATA_DIR / 'wheels', verbose=False)

def record(r, task, t_start: float, t_end: float) -> dict:
    patch = r.agent_patch or ''
    dur = float(r.duration_seconds or 0)
    wall = t_end - t_start
    return {'instance_id': task.instance_id, 'repo': task.repo, 'resolved': bool(r.resolved),
            'status': getattr(r, 'status', None), 'test_exit_code': r.test_exit_code,
            'duration_s': round(dur, 1), 'task_wall_s': round(wall, 1),
            'setup_s': round(max(0.0, wall - dur), 1),   # everything outside the agent loop (setup + grading)  VERIFY
            'tool_calls': getattr(r, 'tool_calls', None), 'llm_calls': getattr(r, 'total_llm_calls', None),
            'total_tokens': getattr(r, 'total_tokens', None), 'error': getattr(r, 'error', None) or getattr(r, 'error_message', None),
            'patch_chars': len(patch), 'patch': patch, 'trace': str(getattr(r, 'trace_path', '') or getattr(r, 'trace_json_path', '') or ''),
            'test_tail': (r.test_output or '')[-1500:], 'max_time_minutes': EVAL.get('max_time_minutes'),
            'max_tool_calls': EVAL.get('max_tool_calls')}

def append_row(path: Path, row: dict) -> None:
    with path.open('a', encoding='utf-8') as fh:
        fh.write(json.dumps(row) + '\n')

In [ ]:
# ---------------------------------------------------------------- EXP-000 gold control (Q5)
# `sound` must exist even when the gold control is skipped, or the agent cell below raises NameError.
sound = list(candidates)
if RUN_GOLD_CONTROL:
    from adk_eval_core.tracing import SessionTrace  # [V] tracing/__init__.py re-exports it; no-arg ctor

    class GoldEvaluator(Evaluator):
        """Phase 1 returns the reference patch; no model call, nothing is shown to any agent."""
        async def _run_agent_sandbox(self, task, *args, **kwargs):  # [V] evaluate.py:194 -> tuple[str, str|None, SessionTrace]
            return task.patch, None, SessionTrace()

    gold = GoldEvaluator(make_config(RUN_DIR / 'gold_control'))
    gold_rows = []
    for i, t in enumerate(candidates, 1):
        t0 = time.time()
        r = run_sync(lambda: gold.evaluate_task(task=t, task_index=i, total_tasks=len(candidates)))
        row = {'instance_id': t.instance_id, 'repo': t.repo, 'mode': 'gold', 'resolved': bool(r.resolved),
               'exit': r.test_exit_code, 'status': getattr(r, 'status', None), 'seconds': round(time.time() - t0, 1), 'error': (getattr(r, 'error', '') or '')[:300],
               'tail': (r.test_output or '')[-600:]}
        gold_rows.append(row)
        append_row(RUN_DIR / 'gold_control.jsonl', row)
        print('gold', t.instance_id, 'resolved=', row['resolved'], f"{row['seconds']}s")
    failures = [x['instance_id'] for x in gold_rows if not x['resolved']]
    (RUN_DIR / 'gold_failures.txt').write_text('# gold-control failures (exclude from cohorts)\n' + '\n'.join(failures) + '\n')
    sound = [t for t, x in zip(candidates, gold_rows) if x['resolved']]
    print(f'gold control: {len(sound)}/{len(candidates)} sound; failures: {failures}')

In [ ]:
# ---------------------------------------------------------------- baseline agent eval
summary = {'exp_id': EXP_ID, 'server_start_s': SERVER_START_S, 'decode_probe': DECODE, 'eval_config': EVAL,
           'compaction': {'interval': COMPACTION_INTERVAL, 'token_threshold': COMPACTION_TOKEN_THRESHOLD},
           'tasks': [], 'resolved': 0}
if RUN_AGENT:
    ev = Evaluator(make_config(RUN_DIR / 'official'))
    for i, t in enumerate(sound, 1):
        t0 = time.time()
        r = run_sync(lambda: ev.evaluate_task(task=t, task_index=i, total_tasks=len(sound)))
        row = record(r, t, t0, time.time())
        append_row(RUN_DIR / 'task_results.jsonl', row)
        (RUN_DIR / 'patches' / f'{t.instance_id}.patch').write_text(row['patch'])
        for cand in (Path(row['trace']) if row['trace'] else None, RUN_DIR / 'official' / 'traces' / f'trace_{t.instance_id}.json'):
            if cand and cand.exists():
                shutil.copy(cand, RUN_DIR / 'traces' / f'trace_{t.instance_id}.json'); break
        summary['tasks'].append(t.instance_id); summary['resolved'] += int(row['resolved'])
        print(f"[{i}/{len(sound)}] {t.instance_id} resolved={row['resolved']} calls={row['tool_calls']} "
              f"agent={row['duration_s']}s setup={row['setup_s']}s err={(row['error'] or '')[:100]}")
    n = len(summary['tasks'])
    print(f"RESOLVED {summary['resolved']}/{n}" if n else 'no tasks run')
(RUN_DIR / 'summary.json').write_text(json.dumps(summary, indent=1))

In [ ]:
# ---------------------------------------------------------------- bundle for download + stop server
# Official logs may contain repository snippets: keep the bundle private, never commit it (plan §1.3).
for extra in (RUN_DIR / 'official' / 'logs',):
    if extra.exists():
        shutil.copytree(extra, RUN_DIR / 'logs', dirs_exist_ok=True)
shutil.make_archive(str(Path('/kaggle/working/runs') / EXP_ID), 'zip', RUN_DIR)
print('bundle:', Path('/kaggle/working/runs') / f'{EXP_ID}.zip')
try:
    server.stop()
except Exception:  # noqa: BLE001
    pass